## 0. Setup Awal

In [1]:
import os
import warnings

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

## 1. Data Understanding

### 1.1 Load Dataset

In [2]:
df = pd.read_csv('burnout_dataset1.csv')

print(f'Jumlah data awal: {len(df)}')

# df = df.sample(
#     n=5000,
#     random_state=42
# ).reset_index(drop=True)
# print(f'Jumlah data setelah sampling: {len(df)}')

Jumlah data awal: 50000


### 1.2 Identifikasi Kolom yang Digunakan

In [3]:
numeric_columns = df.select_dtypes(
    include='number'
).columns

categorical_columns = df.select_dtypes(
    include=['object', 'category']
).columns

print('Kolom Numerik:')
print(
    numeric_columns.tolist()
)

print(
    '\nKolom Kategorikal:'
)
print(
    categorical_columns.tolist()
)

Kolom Numerik:
['Person_ID', 'Age', 'Monthly_Income_USD', 'Work_Hours_Per_Week', 'Job_Satisfaction', 'Work_Life_Balance', 'Sleep_Hours', 'Anxiety_Score', 'Depression_Score', 'Mood_Score', 'Emotional_Stability', 'Physical_Activity_Hours', 'Meditation_Minutes', 'Screen_Time_Hours', 'Social_Media_Hours', 'Gaming_Hours', 'Coffee_Cups_Per_Day', 'Life_Satisfaction', 'Productivity_Score', 'Absenteeism_Days', 'Burnout_Score']

Kolom Kategorikal:
['Gender', 'Country', 'Occupation', 'Education_Level', 'Employment_Status', 'Remote_Work', 'Sleep_Quality', 'Stress_Level', 'Exercise_Frequency', 'Alcohol_Consumption', 'Smoking', 'Healthy_Diet', 'Chronic_Stress', 'Family_History_Mental_Illness', 'Therapy_Attendance', 'Support_System', 'Mental_Health_Status', 'AI_Wellness_Recommendation', 'Burnout_Risk']


## 2. Preprocessing

### 2.1 Seleksi Kolom

In [4]:
y = df['Burnout_Risk'].copy()

numeric_columns = numeric_columns.drop(
    ['Person_ID', 'Burnout_Score'],
    errors='ignore'
)

categorical_columns = categorical_columns.drop(
    [
        'Gender',
        'Country',
        'Occupation',
        'AI_Wellness_Recommendation',
        'Burnout_Risk',
        'Education_Level',
        'Employment_Status'
    ],
    errors='ignore'
)

selected_columns = (
    list(numeric_columns)
    + list(categorical_columns)
)

X = df[
    selected_columns
].copy()

X.head()

,Age,Monthly_Income_USD,Work_Hours_Per_Week,Job_Satisfaction,Work_Life_Balance,Sleep_Hours,Anxiety_Score,Depression_Score,Mood_Score,Emotional_Stability,...,Stress_Level,Exercise_Frequency,Alcohol_Consumption,Smoking,Healthy_Diet,Chronic_Stress,Family_History_Mental_Illness,Therapy_Attendance,Support_System,Mental_Health_Status
0,58.0,2786.0,67,3.0,7.0,4.6,74.0,75.0,26.0,18,...,High,Weekly,Yes,Yes,No,Yes,No,Yes,Average,Critical
1,24.0,8027.0,44,5.0,8.0,4.7,58.0,80.0,29.0,37,...,Moderate,Weekly,Yes,Yes,Yes,No,Yes,No,Good,Critical
2,59.0,3177.0,60,8.0,7.0,6.3,54.0,55.0,44.0,48,...,Moderate,Daily,Yes,No,No,No,Yes,No,Average,Needs Attention
3,65.0,4093.0,25,2.0,9.0,7.1,12.0,11.0,88.0,80,...,Low,Weekly,Yes,No,No,No,No,Yes,Good,Healthy
4,64.0,NaN,52,1.0,4.0,9.1,25.0,48.0,67.0,76,...,Low,Never,Yes,No,No,No,Yes,No,Poor,Healthy


### 2.2 Encoding Data Kategorikal

In [5]:
encoding_maps = {
    'Remote_Work': {
        'No': 0,
        'Hybrid': 1,
        'Yes': 2
    },

    'Sleep_Quality': {
        'Excellent': 0,
        'Good': 1,
        'Average': 2,
        'Poor': 3
    },

    'Stress_Level': {
        'Low': 0,
        'Moderate': 1,
        'High': 2
    },

    'Exercise_Frequency': {
        'Daily': 0,
        'Weekly': 1,
        'Rarely': 2,
        'Never': 3
    },

    'Alcohol_Consumption': {
        'No': 0,
        'Yes': 1
    },

    'Smoking': {
        'No': 0,
        'Yes': 1
    },

    'Healthy_Diet': {
        'No': 1,
        'Yes': 0
    },

    'Chronic_Stress': {
        'No': 0,
        'Yes': 1
    },

    'Family_History_Mental_Illness': {
        'No': 0,
        'Yes': 1
    },

    'Therapy_Attendance': {
        'No': 1,
        'Yes': 0
    },

    'Support_System': {
        'Poor': 3,
        'Average': 2,
        'Good': 1,
        'Excellent': 0
    },

    'Mental_Health_Status': {
        'Healthy': 0,
        'Needs Attention': 1,
        'Critical': 2
    }
}

for column, mapping in encoding_maps.items():

    X[column] = (
        X[column]
        .map(mapping)
    )

target_mapping = {
    'Low': 0,
    'Moderate': 1,
    'High': 2
}

y = y.map(target_mapping)

X[categorical_columns].head()

,Remote_Work,Sleep_Quality,Stress_Level,Exercise_Frequency,Alcohol_Consumption,Smoking,Healthy_Diet,Chronic_Stress,Family_History_Mental_Illness,Therapy_Attendance,Support_System,Mental_Health_Status
0,1,3,2,1,1,1,1,1,0,0.0,2,2
1,1,3,1,1,1,1,0,0,1,1.0,1,2
2,0,2,1,0,1,0,1,0,1,1.0,2,1
3,2,1,0,1,1,0,1,0,0,0.0,1,0
4,0,0,0,3,1,0,1,0,1,1.0,3,0


### 2.3 Penanganan Missing Values

In [6]:
print(
    'Missing value sebelum preprocessing:'
)

print(
    X.isnull().sum()
)

X = X.fillna(
    X.mean()
)

print(
    '\nJumlah missing value setelah preprocessing:'
)

print(
    X.isnull().sum()
)

Missing value sebelum preprocessing:
Age                               250
Monthly_Income_USD               1450
Work_Hours_Per_Week                 0
Job_Satisfaction                  500
Work_Life_Balance                 650
Sleep_Hours                       950
Anxiety_Score                     800
Depression_Score                  850
Mood_Score                        900
Emotional_Stability                 0
Physical_Activity_Hours          1100
Meditation_Minutes               1200
Screen_Time_Hours                 850
Social_Media_Hours                600
Gaming_Hours                      750
Coffee_Cups_Per_Day                 0
Life_Satisfaction                 700
Productivity_Score                950
Absenteeism_Days                    0
Remote_Work                         0
Sleep_Quality                       0
Stress_Level                        0
Exercise_Frequency                  0
Alcohol_Consumption                 0
Smoking                             0
Healthy_Diet 

### 2.4 Penanganan Outlier

### 2.5 Standardisasi

In [7]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X
)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_scaled

,Age,Monthly_Income_USD,Work_Hours_Per_Week,Job_Satisfaction,Work_Life_Balance,Sleep_Hours,Anxiety_Score,Depression_Score,Mood_Score,Emotional_Stability,...,Stress_Level,Exercise_Frequency,Alcohol_Consumption,Smoking,Healthy_Diet,Chronic_Stress,Family_History_Mental_Illness,Therapy_Attendance,Support_System,Mental_Health_Status
0,1.196220,-1.059350,1.497320,-0.876176,0.525001,-1.400175,1.300669,1.318767,-1.315962,-1.586087,...,1.838578,-0.455507,1.008436,1.003687,0.996685,2.53389,-0.996765,-1.012603,0.441976,1.633584
1,-1.261001,0.542844,-0.064238,-0.174969,0.876030,-1.341772,0.680397,1.510455,-1.198546,-0.857858,...,0.411952,-0.455507,1.008436,1.003687,-1.003326,-0.39465,1.003245,0.995518,-0.452894,1.633584
2,1.268491,-0.939820,1.022063,0.876842,0.525001,-0.407328,0.525329,0.552016,-0.611465,-0.436251,...,0.411952,-1.351150,1.008436,-0.996327,0.996685,-0.39465,1.003245,0.995518,0.441976,0.373764
3,1.702118,-0.659795,-1.354221,-1.226780,1.227060,0.059895,-1.102883,-1.134836,1.110641,0.790240,...,-1.014673,-0.455507,1.008436,-0.996327,0.996685,-0.39465,-0.996765,-1.012603,-0.452894,-0.886057
4,1.629847,0.000000,0.478912,-1.577384,-0.528088,1.227951,-0.598913,0.283653,0.288727,0.636928,...,-1.014673,1.335781,1.008436,-0.996327,0.996685,-0.39465,1.003245,0.995518,1.336847,-0.886057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,-0.393747,-0.828849,-0.132132,0.876842,0.173971,0.001492,0.641630,0.321991,-0.572326,-0.551235,...,0.411952,-0.455507,1.008436,-0.996327,-1.003326,-0.39465,1.003245,0.995518,0.441976,0.373764
49996,1.123948,1.338592,-1.422115,-1.226780,0.173971,-1.575383,-1.335485,-1.556549,1.306335,1.556797,...,-1.014673,-1.351150,1.008436,1.003687,0.996685,-0.39465,-0.996765,-1.012603,-1.347764,-0.886057
49997,-0.538289,-0.950214,0.750488,0.876842,-1.230147,0.351909,0.951766,1.433780,-1.041991,-1.011169,...,0.411952,1.335781,1.008436,-0.996327,-1.003326,-0.39465,-0.996765,-1.012603,1.336847,1.633584
49998,1.413033,-0.755175,-1.082646,-0.174969,0.525001,0.001492,-0.327544,-0.904811,0.562698,0.100338,...,-1.014673,1.335781,1.008436,1.003687,0.996685,-0.39465,-0.996765,-1.012603,1.336847,-0.886057


### 2.6 Data Bersih

In [ ]:
print('Dimensi data fitur (X):', X.shape)
print('Dimensi data fitur terstandarisasi (X_scaled):', X_scaled.shape)
print('Dimensi data latih (X_train):', X_train.shape)
print('Dimensi data uji (X_test):', X_test.shape)
print('Jumlah missing value:', X_scaled.isnull().sum().sum())
print('\nDistribusi kelas target (y):')
print(y.value_counts())

X_scaled.head()

X_scaled.to_csv(
    'X_scaled.csv',
    index=False
)

Dimensi data fitur (X): (50000, 31)
Dimensi data fitur terstandarisasi (X_scaled): (50000, 31)
Dimensi data latih (X_train): (40000, 31)
Dimensi data uji (X_test): (10000, 31)
Jumlah missing value: 0

Distribusi kelas target (y):
Burnout_Risk
0    25369
1    14780
2     9851
Name: count, dtype: int64


,Age,Monthly_Income_USD,Work_Hours_Per_Week,Job_Satisfaction,Work_Life_Balance,Sleep_Hours,Anxiety_Score,Depression_Score,Mood_Score,Emotional_Stability,...,Stress_Level,Exercise_Frequency,Alcohol_Consumption,Smoking,Healthy_Diet,Chronic_Stress,Family_History_Mental_Illness,Therapy_Attendance,Support_System,Mental_Health_Status
0,1.196220,-1.059350,1.497320,-0.876176,0.525001,-1.400175,1.300669,1.318767,-1.315962,-1.586087,...,1.838578,-0.455507,1.008436,1.003687,0.996685,2.53389,-0.996765,-1.012603,0.441976,1.633584
1,-1.261001,0.542844,-0.064238,-0.174969,0.876030,-1.341772,0.680397,1.510455,-1.198546,-0.857858,...,0.411952,-0.455507,1.008436,1.003687,-1.003326,-0.39465,1.003245,0.995518,-0.452894,1.633584
2,1.268491,-0.939820,1.022063,0.876842,0.525001,-0.407328,0.525329,0.552016,-0.611465,-0.436251,...,0.411952,-1.351150,1.008436,-0.996327,0.996685,-0.39465,1.003245,0.995518,0.441976,0.373764
3,1.702118,-0.659795,-1.354221,-1.226780,1.227060,0.059895,-1.102883,-1.134836,1.110641,0.790240,...,-1.014673,-0.455507,1.008436,-0.996327,0.996685,-0.39465,-0.996765,-1.012603,-0.452894,-0.886057
4,1.629847,0.000000,0.478912,-1.577384,-0.528088,1.227951,-0.598913,0.283653,0.288727,0.636928,...,-1.014673,1.335781,1.008436,-0.996327,0.996685,-0.39465,1.003245,0.995518,1.336847,-0.886057
